# 01 — Momentum Factor · S&P 500 (2015–2025)

> **QuantAI Research Series** · Notebook reproductible  
> Stack : Python 3.13 · OpenBB Platform (localhost:6900) · yfinance fallback

---

## Objectif

Backtester le **momentum cross-sectionnel** sur le S&P 500 sur la période 2015–2025 et produire les métriques clés :

| Métrique | Description |
|---|---|
| **Sharpe Ratio** | Rendement ajusté au risque (annualisé) |
| **Max Drawdown** | Perte maximale pic-à-creux |
| **IC / t-stat** | Information Coefficient : corrélation signal→retour |
| **Turnover** | Rotation mensuelle du portefeuille |
| **CAGR** | Croissance annuelle composée |

## Définition du signal

Momentum classique **12-1 mois** (Jegadeesh & Titman, 1993) :
$$\text{MOM}_{t} = \frac{P_{t-21}}{P_{t-252}} - 1$$

Skip du dernier mois (21 jours) pour éviter le mean-reversion court terme.  
Portfolio : **Long Q5 / Short Q1** (quintiles cross-sectionnels), rééquilibrage mensuel, equal-weight.

---
## 0 · Setup & Imports

In [ ]:
import orjson  # Force init in main thread before yfinance threads (prevents SIGSEGV on Python 3.13/ARM64)
import warnings
import sys
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Ajouter le root du projet au path
ROOT = Path("__file__").resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signals.factors.momentum import MomentumFactor

pd.set_option("display.float_format", "{:.4f}".format)
np.random.seed(42)

print(f"Python  : {sys.version.split()[0]}")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

---
## 1 · Données — Univers S&P 500 (2015–2025)

In [ ]:
# ── Paramètres globaux ────────────────────────────────────────────────────────
START_DATE      = "2014-01-01"    # +1 an de warm-up
END_DATE        = "2025-03-31"
BACKTEST_START  = "2015-01-01"
LOOKBACK_LONG   = 252             # 12 mois
LOOKBACK_SHORT  = 21              # 1 mois skip
REBAL_FREQ      = "ME"            # Month-End
N_QUINTILES     = 5
RISK_FREE_RATE  = 0.04            # 4% annuel
SAMPLE_SIZE     = 150             # Augmenter à 500 pour l'univers complet

In [ ]:
def load_sp500_constituents() -> list[str]:
    """Récupère les tickers S&P 500 depuis Wikipedia."""
    import io, urllib.request
    req = urllib.request.Request(
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        headers={"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"},
    )
    with urllib.request.urlopen(req) as resp:
        html = resp.read()
    tables = pd.read_html(io.BytesIO(html))
    tickers = tables[0]["Symbol"].tolist()
    return sorted([t.replace(".", "-") for t in tickers])

SP500_TICKERS = load_sp500_constituents()
print(f"Univers S&P 500 : {len(SP500_TICKERS)} tickers")


In [ ]:
import httpx

def fetch_via_openbb(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    """Télécharge les prix via OpenBB Platform (localhost:6900)."""
    OPENBB_URL = "http://localhost:6900"
    all_prices = {}
    with httpx.Client(timeout=30.0) as client:
        for ticker in tickers:
            try:
                resp = client.get(
                    f"{OPENBB_URL}/api/v1/equity/price/historical",
                    params=dict(symbol=ticker, start_date=start, end_date=end, provider="yfinance"),
                )
                if resp.status_code == 200:
                    df = pd.DataFrame(resp.json()["results"])
                    df["date"] = pd.to_datetime(df["date"])
                    all_prices[ticker] = df.set_index("date")["close"]
            except Exception:
                pass
    if not all_prices:
        raise RuntimeError("OpenBB non disponible")
    return pd.DataFrame(all_prices).sort_index()


def fetch_via_yfinance(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    """Fallback direct yfinance."""
    import yfinance as yf
    return yf.download(tickers, start=start, end=end, auto_adjust=True, progress=True, threads=True)["Close"]


def load_prices(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    """Charge les prix via OpenBB, fallback yfinance."""
    print("→ Tentative OpenBB Platform (localhost:6900)...")
    try:
        fetch_via_openbb(tickers[:3], start, end)  # Test connexion
        print("✓ OpenBB disponible")
        prices = fetch_via_openbb(tickers, start, end)
    except Exception as e:
        print(f"⚠ OpenBB indisponible ({e.__class__.__name__}) → fallback yfinance")
        prices = fetch_via_yfinance(tickers, start, end)
    print(f"✓ {prices.shape[0]} jours × {prices.shape[1]} tickers ({prices.index[0].date()} → {prices.index[-1].date()})")
    return prices


# Chargement
tickers_sample = SP500_TICKERS[:SAMPLE_SIZE]
prices_raw = load_prices(tickers_sample, START_DATE, END_DATE)

# Nettoyage
prices = prices_raw.loc[:, prices_raw.notna().mean() >= 0.80].ffill(limit=5)
print(f"Après nettoyage : {prices.shape[1]} tickers retenus")

# Benchmark
spy_prices = load_prices(["SPY"], START_DATE, END_DATE)
spy_returns = spy_prices["SPY"].pct_change().dropna()

---
## 2 · Signal Momentum — Construction

In [ ]:
returns_daily = prices.pct_change()

def compute_momentum_signal(prices: pd.DataFrame, lookback_long: int = 252, lookback_short: int = 21) -> pd.DataFrame:
    """Signal momentum 12-1m : prix[t-skip] / prix[t-lookback] - 1."""
    return prices.shift(lookback_short) / prices.shift(lookback_long) - 1

mom_signal = compute_momentum_signal(prices, LOOKBACK_LONG, LOOKBACK_SHORT)

# Snapshot distribution
snapshot_date = "2020-01-31"
snapshot = mom_signal.loc[:snapshot_date].iloc[-1].dropna()  # handle non-trading-day snapshots

fig = px.histogram(
    snapshot, nbins=30,
    title=f"Distribution du signal Momentum 12-1m (snapshot {snapshot_date})",
    labels={"value": "Momentum Score"},
    color_discrete_sequence=["#1f77b4"],
    template="plotly_white",
)
fig.add_vline(x=snapshot.median(), line_dash="dash", line_color="red",
              annotation_text=f"Médiane: {snapshot.median():.1%}")
fig.update_layout(height=380, showlegend=False)
fig.show()

print(snapshot.describe())

---
## 3 · Backtest — Long/Short Quintiles

In [ ]:
def build_quintile_portfolios(
    signal: pd.DataFrame,
    returns: pd.DataFrame,
    n_quintiles: int = 5,
    rebal_freq: str = "ME",
) -> tuple[dict, pd.DataFrame]:
    """
    Construit les rendements de chaque quintile (equal-weight, rééquilibrage mensuel).
    Retourne : dict de Series de rendements + DataFrame de turnover.
    """
    rebal_dates = signal.resample(rebal_freq).last().index
    rebal_dates = rebal_dates[rebal_dates >= BACKTEST_START]

    q_returns = {f"Q{q}": [] for q in range(1, n_quintiles + 1)}
    q_returns["LS"] = []
    turnover_log = []
    prev_weights = {f"Q{q}": pd.Series(dtype=float) for q in range(1, n_quintiles + 1)}

    for i, rebal_date in enumerate(rebal_dates[:-1]):
        next_rebal = rebal_dates[i + 1]
        sig_t = signal.loc[:rebal_date].iloc[-1].dropna()  # iloc[-1] handles calendar month-ends that fall on weekends
        if len(sig_t) < n_quintiles * 5:
            continue

        quintile_labels = pd.qcut(sig_t, q=n_quintiles, labels=False) + 1
        period_rets = returns.loc[(returns.index > rebal_date) & (returns.index <= next_rebal)]
        if period_rets.empty:
            continue

        q5_ret = q1_ret = None
        for q in range(1, n_quintiles + 1):
            q_tickers = [t for t in quintile_labels[quintile_labels == q].index if t in period_rets.columns]
            if not q_tickers:
                continue
            ret = (1 + period_rets[q_tickers].mean(axis=1)).prod() - 1
            q_returns[f"Q{q}"].append((next_rebal, ret))
            if q == 5: q5_ret = ret
            if q == 1: q1_ret = ret

            # Turnover
            new_w = pd.Series(1 / len(q_tickers), index=q_tickers)
            old_w = prev_weights[f"Q{q}"]
            all_t = new_w.index.union(old_w.index)
            diff = new_w.reindex(all_t).fillna(0) - old_w.reindex(all_t).fillna(0)
            turnover_log.append((next_rebal, f"Q{q}", abs(diff).sum() / 2))
            prev_weights[f"Q{q}"] = new_w

        if q5_ret is not None and q1_ret is not None:
            q_returns["LS"].append((next_rebal, q5_ret - q1_ret))

    result = {}
    for key, vals in q_returns.items():
        if vals:
            dates, rets = zip(*vals)
            result[key] = pd.Series(rets, index=pd.DatetimeIndex(dates), name=key)

    return result, pd.DataFrame(turnover_log, columns=["date", "quintile", "turnover"])


print("Calcul des portfolios quintiles...")
portfolio_returns, turnover_df = build_quintile_portfolios(mom_signal, returns_daily, N_QUINTILES, REBAL_FREQ)
print(f"✓ {len(portfolio_returns)} portfolios construits")
for k, v in portfolio_returns.items():
    print(f"  {k}: {len(v)} périodes")

---
## 4 · Métriques de Performance

In [ ]:
def compute_performance_metrics(returns: pd.Series, rf: float = 0.04, ppy: int = 12) -> dict:
    """Calcule les métriques de performance standard (CAGR, Sharpe, Sortino, Max DD, Calmar)."""
    r = returns.dropna()
    if len(r) < 2:
        return {k: np.nan for k in ["CAGR","Volatility","Sharpe","Sortino","Max Drawdown","Calmar","Hit Rate"]}

    rf_p = rf / ppy
    excess = r - rf_p
    n_years = len(r) / ppy
    total = (1 + r).prod()
    cagr = total ** (1 / n_years) - 1
    vol = r.std() * np.sqrt(ppy)
    sharpe = excess.mean() / excess.std() * np.sqrt(ppy) if excess.std() > 0 else 0
    down_std = r[r < rf_p].std()
    sortino = excess.mean() / down_std * np.sqrt(ppy) if down_std > 0 else 0
    cumret = (1 + r).cumprod()
    max_dd = ((cumret - cumret.cummax()) / cumret.cummax()).min()
    calmar = cagr / abs(max_dd) if max_dd != 0 else 0

    return dict(CAGR=cagr, Volatility=vol, Sharpe=sharpe, Sortino=sortino,
                **{"Max Drawdown": max_dd}, Calmar=calmar, **{"Hit Rate": (r > 0).mean()})


spy_monthly = spy_returns.resample(REBAL_FREQ).apply(lambda x: (1 + x).prod() - 1)
spy_monthly = spy_monthly.loc[spy_monthly.index >= BACKTEST_START]

rows = []
for k, s in {**portfolio_returns, "SPY": spy_monthly}.items():
    m = compute_performance_metrics(s, RISK_FREE_RATE)
    m["Portfolio"] = k
    rows.append(m)

metrics_df = pd.DataFrame(rows).set_index("Portfolio")[
    ["CAGR","Volatility","Sharpe","Sortino","Max Drawdown","Calmar","Hit Rate"]
]

display_df = metrics_df.copy()
for c in ["CAGR","Volatility","Max Drawdown","Hit Rate"]: display_df[c] = metrics_df[c].map("{:.1%}".format)
for c in ["Sharpe","Sortino","Calmar"]: display_df[c] = metrics_df[c].map("{:.2f}".format)

print("\n📊 PERFORMANCE PAR QUINTILE MOMENTUM — S&P 500 (2015–2025)")
print("=" * 72)
print(display_df.to_string())
print("=" * 72)

In [ ]:
# ── Wealth Index ──────────────────────────────────────────────────────────────
colors = {"Q1":"#d62728","Q2":"#ff7f0e","Q3":"#7f7f7f","Q4":"#2ca02c",
          "Q5":"#1f77b4","LS":"#9467bd","SPY":"#000000"}
dashes = {"LS":"dash","SPY":"dot"}

fig = go.Figure()
all_series = {**portfolio_returns, "SPY": spy_monthly.reindex(portfolio_returns["Q1"].index)}

for name, series in all_series.items():
    wealth = (1 + series.fillna(0)).cumprod()
    fig.add_trace(go.Scatter(
        x=wealth.index, y=wealth.values, name=name,
        line=dict(color=colors.get(name,"gray"), dash=dashes.get(name,"solid"),
                  width=2.5 if name in ["LS","Q5","SPY"] else 1.5),
    ))

fig.update_layout(
    title="<b>Momentum Quintiles — Wealth Index (base 1.0)</b><br><sup>Equal-weight · Rééquilibrage mensuel · 2015–2025</sup>",
    xaxis_title="Date", yaxis_title="Valeur (base = 1)",
    template="plotly_white", height=520, hovermode="x unified",
)
fig.show()

In [ ]:
# ── Drawdown L/S ──────────────────────────────────────────────────────────────
ls_r = portfolio_returns["LS"]
ls_cum = (1 + ls_r.fillna(0)).cumprod()
ls_dd = (ls_cum - ls_cum.cummax()) / ls_cum.cummax()

fig_dd = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4],
                       subplot_titles=["L/S Performance cumulée", "Drawdown"])

fig_dd.add_trace(go.Scatter(x=ls_cum.index, y=ls_cum.values,
    fill="tozeroy", fillcolor="rgba(31,119,180,0.1)",
    line=dict(color="#1f77b4", width=2), name="Cumret"), row=1, col=1)

fig_dd.add_trace(go.Scatter(x=ls_dd.index, y=ls_dd.values,
    fill="tozeroy", fillcolor="rgba(214,39,40,0.3)",
    line=dict(color="#d62728", width=1.5), name="Drawdown"), row=2, col=1)

max_dd_date, max_dd_val = ls_dd.idxmin(), ls_dd.min()
fig_dd.add_annotation(x=max_dd_date, y=max_dd_val,
    text=f"Max DD: {max_dd_val:.1%}", showarrow=True, arrowhead=2, row=2, col=1)

fig_dd.update_layout(title="<b>Momentum L/S — Performance & Drawdown</b>",
                     template="plotly_white", height=520, showlegend=False)
fig_dd.show()
print(f"Max Drawdown L/S : {max_dd_val:.1%} ({max_dd_date.date()})")

---
## 5 · Information Coefficient (IC) & t-stat

In [ ]:
def compute_ic_series(
    signal: pd.DataFrame,
    forward_returns: pd.DataFrame,
    rebal_freq: str = "ME",
    method: str = "spearman",
) -> pd.Series:
    """
    Calcule la série d'IC (Information Coefficient) : corrélation(signal_t, retour_forward_t).
    Méthode Spearman recommandée (robuste aux outliers).
    """
    rebal_dates = signal.resample(rebal_freq).last().index
    rebal_dates = rebal_dates[rebal_dates >= BACKTEST_START]
    ic_values = []

    for i, date in enumerate(rebal_dates[:-1]):
        next_date = rebal_dates[i + 1]
        sig = signal.loc[:date].iloc[-1].dropna()  # handle calendar month-end on weekends
        period_rets = forward_returns.loc[(forward_returns.index > date) & (forward_returns.index <= next_date)]
        fwd_ret = (1 + period_rets).prod() - 1
        common = sig.index.intersection(fwd_ret.dropna().index)
        if len(common) < 20:
            continue
        s, f = sig.loc[common], fwd_ret.loc[common]
        corr = stats.spearmanr(s, f)[0] if method == "spearman" else s.corr(f)
        ic_values.append((next_date, corr))

    if not ic_values:
        return pd.Series(dtype=float)
    dates, ics = zip(*ic_values)
    return pd.Series(ics, index=pd.DatetimeIndex(dates), name="IC")


ic_series = compute_ic_series(mom_signal, returns_daily, REBAL_FREQ)

ic_mean = ic_series.mean()
ic_std  = ic_series.std()
ic_ir   = ic_mean / ic_std
t_stat, p_value = stats.ttest_1samp(ic_series.dropna(), 0)
n_obs = len(ic_series.dropna())

print("📐 INFORMATION COEFFICIENT (IC) — Momentum 12-1m")
print("=" * 50)
print(f"  IC moyen        : {ic_mean:+.4f}")
print(f"  IC std          : {ic_std:.4f}")
print(f"  IC IR (ICIR)    : {ic_ir:.3f}")
print(f"  t-statistique   : {t_stat:.3f}")
print(f"  p-value         : {p_value:.4f}  {'✓ Sig. 5%' if p_value < 0.05 else '✗ Non sig.'}")
print(f"  N observations  : {n_obs}")
print(f"  IC > 0 (%)      : {(ic_series > 0).mean():.1%}")
print("=" * 50)

In [ ]:
# ── Visualisation IC ──────────────────────────────────────────────────────────
ic_ma6 = ic_series.rolling(6).mean()

fig_ic = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35],
    subplot_titles=[f"IC mensuel Spearman — Moy: {ic_mean:+.3f} | t-stat: {t_stat:.2f} | p: {p_value:.3f}",
                    "IC cumulé"])

fig_ic.add_trace(go.Bar(x=ic_series.index, y=ic_series.values,
    marker_color=["#d62728" if v < 0 else "#1f77b4" for v in ic_series.values],
    name="IC mensuel", opacity=0.7), row=1, col=1)

fig_ic.add_trace(go.Scatter(x=ic_ma6.index, y=ic_ma6.values,
    line=dict(color="black", width=2.5), name="MA 6m"), row=1, col=1)

fig_ic.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
fig_ic.add_hline(y=ic_mean, line_dash="dot", line_color="steelblue",
                 annotation_text=f"Moy={ic_mean:+.3f}", annotation_position="right", row=1, col=1)

fig_ic.add_trace(go.Scatter(x=ic_series.index, y=ic_series.cumsum().values,
    fill="tozeroy", fillcolor="rgba(31,119,180,0.15)",
    line=dict(color="#1f77b4", width=2), name="IC cumulé"), row=2, col=1)

fig_ic.update_layout(title="<b>Information Coefficient — Momentum 12-1m · S&P 500</b>",
                     template="plotly_white", height=560)
fig_ic.show()

---
## 6 · Analyse Saisonnière & Performance Annuelle

In [ ]:
# ── IC par mois ───────────────────────────────────────────────────────────────
month_names = ["Jan","Fév","Mar","Avr","Mai","Jun","Jul","Aoû","Sep","Oct","Nov","Déc"]
ic_by_month = ic_series.groupby(ic_series.index.month).mean()
ic_by_month.index = month_names

fig_seas = px.bar(ic_by_month.reset_index(), x="index", y="IC",
    color="IC", color_continuous_scale="RdBu", color_continuous_midpoint=0,
    title="<b>Saisonnalité de l'IC Momentum</b>",
    labels={"index": "Mois", "IC": "IC Moyen"}, template="plotly_white")
fig_seas.add_hline(y=0, line_dash="dash", line_color="black")
fig_seas.add_hline(y=ic_mean, line_dash="dot", line_color="steelblue", annotation_text="Moy globale")
fig_seas.update_layout(height=380, coloraxis_showscale=False)
fig_seas.show()

In [ ]:
# ── Performance annuelle L/S vs SPY ──────────────────────────────────────────
ls_annual  = portfolio_returns["LS"].resample("YE").apply(lambda x: (1+x).prod()-1)
spy_annual = spy_monthly.resample("YE").apply(lambda x: (1+x).prod()-1)
annual_df  = pd.DataFrame({"L/S Momentum": ls_annual, "SPY": spy_annual.reindex(ls_annual.index)})
annual_df.index = annual_df.index.year

fig_ann = go.Figure()
fig_ann.add_trace(go.Bar(x=annual_df.index, y=annual_df["L/S Momentum"], name="L/S Momentum",
    marker_color=["#d62728" if v < 0 else "#1f77b4" for v in annual_df["L/S Momentum"]]))
fig_ann.add_trace(go.Scatter(x=annual_df.index, y=annual_df["SPY"],
    mode="markers+lines", name="SPY",
    line=dict(color="black", width=2, dash="dot"), marker=dict(size=8)))
fig_ann.add_hline(y=0, line_color="gray", line_dash="dash")
fig_ann.update_layout(title="<b>Performance Annuelle — L/S Momentum vs SPY</b>",
    yaxis_tickformat=".0%", template="plotly_white", height=400)
fig_ann.show()

print(annual_df.map("{:.1%}".format).to_string())

---
## 7 · Turnover & Coûts de Transaction

In [ ]:
TRANSACTION_COST_BPS = 10  # 10 bps aller-retour

avg_turnover_ls = turnover_df[turnover_df["quintile"].isin(["Q1","Q5"])]["turnover"].mean()
monthly_cost = avg_turnover_ls * TRANSACTION_COST_BPS / 10000
ls_returns_net = portfolio_returns["LS"] - monthly_cost
metrics_net = compute_performance_metrics(ls_returns_net, RISK_FREE_RATE)
ls_metrics  = compute_performance_metrics(portfolio_returns["LS"], RISK_FREE_RATE)

print(f"🔄 Turnover mensuel moyen L/S : {avg_turnover_ls:.1%}")
print(f"   Coût mensuel ({TRANSACTION_COST_BPS}bps AR)   : {monthly_cost*10000:.1f} bps")
print()
print(f"   Sharpe brut  : {ls_metrics['Sharpe']:.3f}")
print(f"   Sharpe net   : {metrics_net['Sharpe']:.3f}")
print(f"   CAGR brut    : {ls_metrics['CAGR']:.1%}")
print(f"   CAGR net     : {metrics_net['CAGR']:.1%}")

---
## 8 · Spread des Quintiles

In [ ]:
bar_colors = ["#d62728","#ff9896","#aec7e8","#98df8a","#1f77b4"]
q_labels = [k for k in portfolio_returns if k.startswith("Q")]
q_cagr   = [compute_performance_metrics(portfolio_returns[k], RISK_FREE_RATE)["CAGR"] for k in q_labels]
q_sharpe = [compute_performance_metrics(portfolio_returns[k], RISK_FREE_RATE)["Sharpe"] for k in q_labels]
spy_m    = compute_performance_metrics(spy_monthly, RISK_FREE_RATE)

fig_bars = make_subplots(rows=1, cols=2, subplot_titles=["CAGR par quintile", "Sharpe par quintile"])
fig_bars.add_trace(go.Bar(x=q_labels, y=q_cagr, marker_color=bar_colors,
    text=[f"{v:.1%}" for v in q_cagr], textposition="outside", name="CAGR"), row=1, col=1)
fig_bars.add_trace(go.Bar(x=q_labels, y=q_sharpe, marker_color=bar_colors,
    text=[f"{v:.2f}" for v in q_sharpe], textposition="outside", name="Sharpe"), row=1, col=2)
fig_bars.add_hline(y=spy_m["CAGR"], line_dash="dot", annotation_text="SPY", row=1, col=1)
fig_bars.add_hline(y=spy_m["Sharpe"], line_dash="dot", annotation_text="SPY", row=1, col=2)
fig_bars.update_yaxes(tickformat=".0%", row=1, col=1)
fig_bars.update_layout(
    title="<b>Spread quintiles momentum — Monotonie attendue Q1 → Q5</b>",
    template="plotly_white", height=420, showlegend=False)
fig_bars.show()

---
## 9 · Analyse de Sensibilité (Lookback)

In [ ]:
configs = [
    (126, 21, "6m-1m"),
    (189, 21, "9m-1m"),
    (252, 21, "12m-1m ★"),
    (252,  0, "12m-0m (no skip)"),
    (378, 21, "18m-1m"),
]

rows = []
for lb_long, lb_short, label in configs:
    sig = compute_momentum_signal(prices, lb_long, lb_short)
    ic  = compute_ic_series(sig, returns_daily, REBAL_FREQ)
    t, p = stats.ttest_1samp(ic.dropna(), 0)
    port, _ = build_quintile_portfolios(sig, returns_daily, N_QUINTILES, REBAL_FREQ)
    ls_m = compute_performance_metrics(port.get("LS", pd.Series(dtype=float)), RISK_FREE_RATE)
    rows.append(dict(Config=label, **{"IC Moy": ic.mean(), "t-stat": t,
        "ICIR": ic.mean()/ic.std() if ic.std()>0 else 0,
        "Sharpe L/S": ls_m["Sharpe"], "CAGR L/S": ls_m["CAGR"], "Max DD": ls_m["Max Drawdown"]}))

sens_df = pd.DataFrame(rows).set_index("Config")

print("🔬 SENSIBILITÉ À LA FENÊTRE LOOKBACK")
d = sens_df.copy()
for c in ["IC Moy","CAGR L/S","Max DD"]: d[c] = sens_df[c].map("{:+.3f}".format)
for c in ["t-stat","ICIR","Sharpe L/S"]: d[c] = sens_df[c].map("{:.3f}".format)
print(d.to_string())

---
## 10 · Résumé Exécutif & Export

In [ ]:
spy_metrics = compute_performance_metrics(spy_monthly, RISK_FREE_RATE)

print("═" * 62)
print("  RÉSUMÉ — MOMENTUM FACTOR BACKTEST (2015–2025)")
print("  S&P 500 · Signal 12-1m · Rééquilibrage mensuel")
print("═" * 62)
print()
print("  SIGNAL")
print(f"    IC moyen (Spearman)  : {ic_mean:+.4f}")
print(f"    t-statistique        : {t_stat:.3f}  (p={p_value:.4f})")
print(f"    ICIR                 : {ic_ir:.3f}")
print()
print("  PORTEFEUILLE L/S (Q5 − Q1)")
print(f"    CAGR brut            : {ls_metrics['CAGR']:+.1%}")
print(f"    Sharpe brut          : {ls_metrics['Sharpe']:.3f}")
print(f"    Sharpe net (10bps)   : {metrics_net['Sharpe']:.3f}")
print(f"    Max Drawdown         : {ls_metrics['Max Drawdown']:.1%}")
print(f"    Sortino              : {ls_metrics['Sortino']:.3f}")
print(f"    Calmar               : {ls_metrics['Calmar']:.3f}")
print()
print("  BENCHMARK SPY")
print(f"    CAGR                 : {spy_metrics['CAGR']:+.1%}")
print(f"    Sharpe               : {spy_metrics['Sharpe']:.3f}")
print(f"    Max Drawdown         : {spy_metrics['Max Drawdown']:.1%}")
print()
print("  VERDICT")
if t_stat > 2.0 and ic_ir > 0.3:
    print("    ✓ Signal statistiquement robuste → intégration QuantAI recommandée")
elif t_stat > 1.5:
    print("    ⚠ Signal marginal → combiner avec quality/value (SignalVector)")
else:
    print("    ✗ Signal faible → réviser définition ou élargir l'univers")
print()
print("  NEXT STEPS")
print("    1. Intégrer via signals/aggregator.py (SignalVector)")
print("    2. Factor neutralisation (beta, secteur, market cap)")
print("    3. Notebook 02 — MiroFish stress scenarios")
print("═" * 62)

In [ ]:
import json
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

export = {
    "meta": {"backtest_start": BACKTEST_START, "end_date": END_DATE,
              "universe": "S&P500", "signal": "momentum_12_1m", "n_tickers": int(prices.shape[1])},
    "ic":   {"mean": float(ic_mean), "std": float(ic_std), "icir": float(ic_ir),
              "t_stat": float(t_stat), "p_value": float(p_value), "n_obs": int(n_obs)},
    "long_short": {"cagr_gross": float(ls_metrics["CAGR"]), "sharpe_gross": float(ls_metrics["Sharpe"]),
                   "sharpe_net": float(metrics_net["Sharpe"]), "max_drawdown": float(ls_metrics["Max Drawdown"]),
                   "sortino": float(ls_metrics["Sortino"]), "calmar": float(ls_metrics["Calmar"])},
    "spy":  {"cagr": float(spy_metrics["CAGR"]), "sharpe": float(spy_metrics["Sharpe"]),
              "max_drawdown": float(spy_metrics["Max Drawdown"])},
}

with open(RESULTS_DIR / "momentum_backtest_results.json", "w") as f:
    json.dump(export, f, indent=2)

pd.DataFrame(portfolio_returns).to_parquet(RESULTS_DIR / "quintile_returns.parquet")
ic_series.to_frame().to_parquet(RESULTS_DIR / "ic_series.parquet")

print(f"✓ Résultats exportés dans research/results/")
print(f"  - momentum_backtest_results.json")
print(f"  - quintile_returns.parquet")
print(f"  - ic_series.parquet")

---

## Références

- Jegadeesh, N., & Titman, S. (1993). *Returns to Buying Winners and Selling Losers*. Journal of Finance, 48(1), 65–91.
- Asness, C., Moskowitz, T., & Pedersen, L. (2013). *Value and Momentum Everywhere*. Journal of Finance, 68(3), 929–985.
- Fama, E. & French, K. (2012). *Size, Value, and Momentum in International Stock Returns*. Journal of Financial Economics.
- Grinold, R. & Kahn, R. (2000). *Active Portfolio Management*. McGraw-Hill.

---
*QuantAI Research · [github.com/Ouo-droid/quantai](https://github.com/Ouo-droid/quantai)*